# 04 — Purchase Recommendation & Sensitivity Analysis
**DRM Key Demand Forecasting**

This notebook translates the 30-day forecast into a concrete DRM key purchase recommendation with multiple buffer scenarios.

---

**Deliverables:**
1. Monthly purchase recommendation (point estimate + confidence interval)
2. Sensitivity analysis: 5%, 10%, 15% buffer scenarios
3. Cost-vs-shortage trade-off analysis
4. Weekly breakdown for phased procurement
5. Executive summary table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'figure.dpi': 100,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print('Libraries loaded.')

In [ ]:
DATA_DIR = os.path.join('..', 'Dataset')

# Load forecast from previous notebook
df_forecast = pd.read_csv(
    os.path.join(DATA_DIR, 'forecast_30d.csv'),
    parse_dates=['Date']
)

# Load historical data for context
df_hist = pd.read_csv(
    os.path.join(DATA_DIR, 'daily_drm_keys_clean.csv'),
    parse_dates=['LogDate']
)

# Load model evaluation
df_eval = pd.read_csv(os.path.join(DATA_DIR, 'model_evaluation.csv'))

print(f'Forecast period: {df_forecast["Date"].min().date()} → {df_forecast["Date"].max().date()}')
print(f'Historical data: {len(df_hist)} days')
display(df_forecast.head())

## 1. Base Purchase Recommendation

In [ ]:
total_forecast      = df_forecast['Forecast'].sum()
total_lower_90      = df_forecast['Lower_90'].sum()
total_upper_90      = df_forecast['Upper_90'].sum()
avg_daily_forecast  = df_forecast['Forecast'].mean()
max_daily_forecast  = df_forecast['Forecast'].max()
min_daily_forecast  = df_forecast['Forecast'].min()

print('=' * 60)
print('30-DAY DRM KEY PURCHASE — BASE FORECAST')
print('=' * 60)
print(f'Total forecast (point)     : {total_forecast:>12,} keys')
print(f'90% CI lower bound         : {total_lower_90:>12,} keys')
print(f'90% CI upper bound         : {total_upper_90:>12,} keys')
print(f'Average daily forecast     : {avg_daily_forecast:>12,.0f} keys/day')
print(f'Peak daily forecast        : {max_daily_forecast:>12,} keys/day')
print(f'Min daily forecast         : {min_daily_forecast:>12,} keys/day')
print('=' * 60)

## 2. Sensitivity Analysis — Buffer Scenarios

Real-world procurement requires a buffer above the point forecast to account for:
- Model uncertainty
- Unexpected demand spikes (special events, promotions)
- Supply-side delays

We evaluate 5%, 10%, and 15% buffer margins.

In [ ]:
buffer_pcts = [0, 5, 10, 15, 20]

scenarios = []
for buf in buffer_pcts:
    purchase_qty = int(np.ceil(total_forecast * (1 + buf / 100)))
    surplus_vs_forecast = purchase_qty - total_forecast
    coverage_of_upper_ci = purchase_qty / total_upper_90 * 100 if total_upper_90 > 0 else 0

    scenarios.append({
        'Buffer (%)': f'{buf}%',
        'Purchase Qty': purchase_qty,
        'Surplus vs Forecast': surplus_vs_forecast,
        'Coverage of 90% CI Upper': f'{coverage_of_upper_ci:.1f}%',
        'Risk Level': (
            'High — likely shortage' if buf == 0 else
            'Medium — some risk' if buf <= 5 else
            'Low — balanced' if buf <= 10 else
            'Very Low — conservative' if buf <= 15 else
            'Minimal — max safety'
        )
    })

df_scenarios = pd.DataFrame(scenarios)
print('BUFFER SCENARIO ANALYSIS')
print('=' * 80)
display(df_scenarios)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scenario comparison bar chart
ax1 = axes[0]
colors_bar = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db', '#9b59b6']
purchase_qtys = [s['Purchase Qty'] for s in scenarios]
bars = ax1.bar([s['Buffer (%)'] for s in scenarios], purchase_qtys, color=colors_bar, edgecolor='white')
ax1.axhline(total_forecast, color='black', linestyle='--', alpha=0.5, label='Point forecast')
ax1.axhline(total_upper_90, color='crimson', linestyle=':', alpha=0.5, label='90% CI upper')
ax1.set_title('Purchase Quantity by Buffer Scenario')
ax1.set_ylabel('Total Keys (30 days)')
ax1.set_xlabel('Buffer %')
ax1.legend()
for bar, qty in zip(bars, purchase_qtys):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + total_forecast * 0.005,
             f'{qty:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Risk vs cost trade-off
ax2 = axes[1]
surplus_vals = [s['Surplus vs Forecast'] for s in scenarios]
ax2.plot(buffer_pcts, surplus_vals, 'o-', color='steelblue', markersize=8, linewidth=2)
ax2.fill_between(buffer_pcts, 0, surplus_vals, alpha=0.2, color='steelblue')
ax2.set_title('Surplus Keys (Potential Waste) by Buffer')
ax2.set_xlabel('Buffer (%)')
ax2.set_ylabel('Surplus Keys')
for x, y in zip(buffer_pcts, surplus_vals):
    ax2.annotate(f'{y:,}', (x, y), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Cost-vs-Shortage Trade-off

Quantify the financial impact of each scenario. Adjust `COST_PER_KEY` and `SHORTAGE_PENALTY_MULTIPLIER` to your actual business parameters.

In [ ]:
# === BUSINESS PARAMETERS — adjust to your actual contract terms ===
COST_PER_KEY = 1.0              # Unit cost per DRM key (normalize to 1 if confidential)
SHORTAGE_PENALTY_MULTIPLIER = 3  # Cost multiplier for emergency/spot purchase

# Simulate scenarios using the 90% CI upper bound as "realistic worst case"
realistic_demand = total_upper_90

cost_analysis = []
for buf in buffer_pcts:
    purchase = int(np.ceil(total_forecast * (1 + buf / 100)))
    
    if purchase >= realistic_demand:
        waste = purchase - realistic_demand
        shortage = 0
    else:
        waste = 0
        shortage = realistic_demand - purchase
    
    base_cost = purchase * COST_PER_KEY
    waste_cost = waste * COST_PER_KEY
    shortage_cost = shortage * COST_PER_KEY * SHORTAGE_PENALTY_MULTIPLIER
    total_cost = base_cost + shortage_cost

    cost_analysis.append({
        'Buffer': f'{buf}%',
        'Purchase': purchase,
        'Potential Waste': waste,
        'Potential Shortage': shortage,
        'Base Cost': base_cost,
        'Shortage Penalty': shortage_cost,
        'Total Expected Cost': total_cost
    })

df_cost = pd.DataFrame(cost_analysis)
print('COST-vs-SHORTAGE ANALYSIS')
print(f'(Assuming worst-case demand = 90% CI upper bound: {realistic_demand:,})')
print('=' * 90)
display(df_cost)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(buffer_pcts))
width = 0.35

bars1 = ax.bar(x - width/2, df_cost['Base Cost'], width, label='Base Cost', color='steelblue')
bars2 = ax.bar(x + width/2, df_cost['Shortage Penalty'], width, label='Shortage Penalty', color='crimson')

ax.plot(x, df_cost['Total Expected Cost'], 'D-', color='black', markersize=8, linewidth=2, label='Total Cost')

ax.set_xlabel('Buffer Scenario')
ax.set_ylabel('Cost (normalized units)')
ax.set_title('Cost Breakdown by Buffer Scenario')
ax.set_xticks(x)
ax.set_xticklabels([f'{b}%' for b in buffer_pcts])
ax.legend()

plt.tight_layout()
plt.show()

optimal_idx = df_cost['Total Expected Cost'].idxmin()
print(f'\n→ Optimal buffer: {df_cost.loc[optimal_idx, "Buffer"]} (minimizes total expected cost)')

## 4. Weekly Breakdown — Phased Procurement

In [ ]:
df_forecast['Week'] = df_forecast['Date'].dt.isocalendar().week.astype(int)
df_forecast['DayOfWeek'] = df_forecast['Date'].dt.day_name()

weekly_summary = (
    df_forecast.groupby('Week').agg(
        Start=('Date', 'min'),
        End=('Date', 'max'),
        Days=('Date', 'count'),
        Total_Forecast=('Forecast', 'sum'),
        Avg_Daily=('Forecast', 'mean'),
        Peak_Day=('Forecast', 'max'),
        Lower_90=('Lower_90', 'sum'),
        Upper_90=('Upper_90', 'sum')
    ).reset_index()
)

weekly_summary['Start'] = weekly_summary['Start'].dt.strftime('%Y-%m-%d')
weekly_summary['End'] = weekly_summary['End'].dt.strftime('%Y-%m-%d')
weekly_summary['Avg_Daily'] = weekly_summary['Avg_Daily'].round(0).astype(int)

print('WEEKLY FORECAST BREAKDOWN')
print('=' * 90)
display(weekly_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

weeks = weekly_summary['Week'].astype(str)
ax.bar(weeks, weekly_summary['Total_Forecast'], color='steelblue', edgecolor='white', label='Forecast')
ax.errorbar(
    weeks, weekly_summary['Total_Forecast'],
    yerr=[
        weekly_summary['Total_Forecast'] - weekly_summary['Lower_90'],
        weekly_summary['Upper_90'] - weekly_summary['Total_Forecast']
    ],
    fmt='none', color='black', capsize=5, label='90% CI'
)
ax.set_title('Weekly DRM Key Forecast with Confidence Intervals')
ax.set_xlabel('Week Number')
ax.set_ylabel('Total Keys')
ax.legend()

for i, (w, total) in enumerate(zip(weeks, weekly_summary['Total_Forecast'])):
    ax.text(i, total + total * 0.01, f'{total:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Historical Context — Forecast vs Recent Actuals

In [ ]:
# Compare forecast with last 3 months' actuals
last_90_days = df_hist.tail(90)
hist_monthly = (
    last_90_days
    .assign(Period=last_90_days['LogDate'].dt.to_period('M'))
    .groupby('Period')['Total_Daily_Keys']
    .agg(['sum', 'mean', 'max'])
    .reset_index()
)
hist_monthly.columns = ['Period', 'Total_Keys', 'Avg_Daily', 'Peak_Daily']

print('HISTORICAL CONTEXT — Last 3 Months')
display(hist_monthly)

print(f'\nForecast next 30 days : {total_forecast:>10,} keys (total)')
print(f'Last 30 days actual   : {last_90_days.tail(30)["Total_Daily_Keys"].sum():>10,} keys (total)')
change = (total_forecast / last_90_days.tail(30)['Total_Daily_Keys'].sum() - 1) * 100
print(f'Forecast vs last month: {change:>+10.1f}%')

## 6. Executive Summary

In [ ]:
RECOMMENDED_BUFFER = 10  # Adjust based on cost analysis above

recommended_purchase = int(np.ceil(total_forecast * (1 + RECOMMENDED_BUFFER / 100)))

print('╔' + '═' * 62 + '╗')
print('║' + ' EXECUTIVE SUMMARY — DRM KEY PURCHASE RECOMMENDATION '.center(62) + '║')
print('╠' + '═' * 62 + '╣')
print('║' + ''.center(62) + '║')
print('║' + f'  Forecast Period : {df_forecast["Date"].min().date()} → {df_forecast["Date"].max().date()}'.ljust(62) + '║')
print('║' + f'  Forecasting Model : Prophet (retrained on full data)'.ljust(62) + '║')
print('║' + f'  Model MAPE (test) : {df_eval["MAPE (%)"].min():.2f}%'.ljust(62) + '║')
print('║' + ''.center(62) + '║')
print('║' + '  ─── FORECAST ───'.ljust(62) + '║')
print('║' + f'  Point estimate     : {total_forecast:>10,} keys'.ljust(62) + '║')
print('║' + f'  90% CI lower       : {total_lower_90:>10,} keys'.ljust(62) + '║')
print('║' + f'  90% CI upper       : {total_upper_90:>10,} keys'.ljust(62) + '║')
print('║' + ''.center(62) + '║')
print('║' + '  ─── RECOMMENDATION ───'.ljust(62) + '║')
print('║' + f'  Buffer applied     : {RECOMMENDED_BUFFER}%'.ljust(62) + '║')
print('║' + f'  ★ PURCHASE QTY     : {recommended_purchase:>10,} keys'.ljust(62) + '║')
print('║' + f'  Avg daily          : {recommended_purchase // 30:>10,} keys/day'.ljust(62) + '║')
print('║' + ''.center(62) + '║')
print('╚' + '═' * 62 + '╝')

In [ ]:
# Final visualization — full picture
fig, ax = plt.subplots(figsize=(16, 6))

# Historical
recent_60 = df_hist.tail(60)
ax.plot(recent_60['LogDate'], recent_60['Total_Daily_Keys'],
        '-', color='steelblue', linewidth=1.5, label='Historical (actual)', alpha=0.7)
ax.plot(recent_60['LogDate'], recent_60['MA_7'],
        '-', color='navy', linewidth=2, label='7-day MA (actual)')

# Forecast
ax.plot(df_forecast['Date'], df_forecast['Forecast'],
        'o-', color='darkorange', markersize=4, linewidth=2, label='Forecast')
ax.fill_between(
    df_forecast['Date'],
    df_forecast['Lower_90'],
    df_forecast['Upper_90'],
    alpha=0.2, color='darkorange', label='90% CI'
)

# Recommended purchase level
ax.axhline(
    recommended_purchase / 30,
    color='green', linestyle='--', linewidth=2, alpha=0.7,
    label=f'Recommended daily budget ({recommended_purchase // 30:,}/day)'
)

# Split line
ax.axvline(df_hist['LogDate'].max(), color='black', linestyle=':', alpha=0.5)
ax.text(df_hist['LogDate'].max(), ax.get_ylim()[1] * 0.98, ' Forecast →',
        fontsize=10, va='top', ha='left', fontweight='bold')

ax.set_title('DRM Key Demand — Historical + 30-Day Forecast with Purchase Recommendation', fontsize=14)
ax.set_ylabel('DRM Keys per Day')
ax.legend(loc='upper left', fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Export recommendation
OUTPUT_DIR = os.path.join('..', 'Dataset')

recommendation = {
    'forecast_period_start': df_forecast['Date'].min().strftime('%Y-%m-%d'),
    'forecast_period_end': df_forecast['Date'].max().strftime('%Y-%m-%d'),
    'total_forecast_point': int(total_forecast),
    'total_forecast_lower_90': int(total_lower_90),
    'total_forecast_upper_90': int(total_upper_90),
    'recommended_buffer_pct': RECOMMENDED_BUFFER,
    'recommended_purchase_qty': recommended_purchase,
    'model': 'Prophet',
    'model_mape_pct': float(df_eval['MAPE (%)'].min())
}

pd.DataFrame([recommendation]).to_csv(
    os.path.join(OUTPUT_DIR, 'purchase_recommendation.csv'), index=False
)
df_scenarios.to_csv(
    os.path.join(OUTPUT_DIR, 'buffer_scenarios.csv'), index=False
)

print('Exported:')
print(f'  → purchase_recommendation.csv')
print(f'  → buffer_scenarios.csv')
print(f'\n✓ Pipeline complete.')